**Dataset :** dair-ai/emotion | **Model :** distilbert-base-uncased  | **Task: **6-class emotion classification

In [28]:
# To install the required packages
# Transformers: HuggingFace model + Trainer API
# To load the datasets
# wandb: model tracking
# scikit-learn: For accuracy metrics
!pip3 install -q transformers datasets wandb scikit-learn huggingface_hub

In [29]:
# Authenticate with W&B and Hugging Face
# Secrets are stored in Kaggle: Add-ons → Secrets (never hardcode tokens)
# Get the the secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

secrets = UserSecretsClient()

WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

wandb.login(key=WANDB_API_KEY)
login(token=HF_TOKEN)

print ("Authentication works with WANDB and HF")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Authentication works with WANDB and HF


In [30]:
# To load the dair-ai/emotion dataset
# 20,000 English tweets labelled with 6 emotions
# train(16k) / validation(2k) / test(2k)
from datasets import load_dataset
from collections import Counter

raw = load_dataset("dair-ai/emotion")
print(raw)

# Map id to label and label to id mappings from dataset schema
label_names = raw["train"].features["label"].names
id2label    = {i: name for i, name in enumerate(label_names)}
label2id    = {name: i for i, name in enumerate(label_names)}
NUM_LABELS  = len(id2label)

print("Labels:", id2label)
print("\nClass distribution (train):")
for lid, cnt in sorted(Counter(raw["train"]["label"]).items()):
    print(f"  {id2label[lid]:10s}: {cnt}")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
Labels: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

Class distribution (train):
  sadness   : 4666
  joy       : 5362
  love      : 1304
  anger     : 2159
  fear      : 1937
  surprise  : 572


In [31]:
# Save id2label.json
# Only this mapping file is committed to GitHub — not the full dataset
import json
with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)

print("id2label.json is saved")
print(json.dumps(id2label, indent=2))

id2label.json is saved
{
  "0": "sadness",
  "1": "joy",
  "2": "love",
  "3": "anger",
  "4": "fear",
  "5": "surprise"
}


In [32]:
# Cleaning the dataset 
# Lowercase  — DistilBERT-uncased expects lowercase input
# Strip URLs — carry no emotional signal
# Strip @mentions & #hashtags 
# Collapse whitespace — normalises newline/multi-space artefacts

import re

def clean_text(example):
    text = example["text"].lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return {"text": text}

cleaned = raw.map(clean_text)

# Remove any rows where text became empty after cleaning
cleaned = cleaned.filter(lambda x: len(x["text"]) > 0)

print(f"Train after cleaning: {len(cleaned['train'])} samples")
print(f"Test  after cleaning: {len(cleaned['test'])}  samples")

Train after cleaning: 16000 samples
Test  after cleaning: 2000  samples


In [33]:
# Tokenisation 
from transformers import DistilBertTokenizerFast
model_name = "distilbert-base-uncased"
max_length = 512

tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)


# Extract texts and labels from each split
train_texts  = list(cleaned["train"]["text"])
train_labels = [int(y) for y in cleaned["train"]["label"]]
test_texts   = list(cleaned["test"]["text"])
test_labels  = [int(y) for y in cleaned["test"]["label"]]
val_texts    = list(cleaned["validation"]["text"])
val_labels   = [int(y) for y in cleaned["validation"]["label"]]


# Tokenise train,test,validation
train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=max_length)
test_enc  = tokenizer(test_texts,  truncation=True, padding=True, max_length=max_length)
val_enc   = tokenizer(val_texts,   truncation=True, padding=True, max_length=max_length)

train_labels_encoded = [int(y) for y in train_labels]
test_labels_encoded  = [int(y) for y in test_labels]

print(f"Train encodings: {len(train_enc['input_ids'])} samples")
print(f"Test  encodings: {len(test_enc['input_ids'])}  samples")
print(f"Test  encodings: {len(val_enc['input_ids'])}  samples")
print(f"\nSample token preview:")
print(' '.join(train_enc[1].tokens[:200]))

Train encodings: 16000 samples
Test  encodings: 2000  samples
Test  encodings: 2000  samples

Sample token preview:
[CLS] i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [46]:
# What BERT actually USES (numerical IDs)
print("\nInput IDs (numbers):")
print(train_enc[1].ids[:20])

# Attention mask (1 = real token, 0 = padding)
print("\nAttention mask:")
print(train_enc[1].attention_mask[:20])


Input IDs (numbers):
[101, 1045, 2064, 2175, 2013, 3110, 2061, 20625, 2000, 2061, 9636, 17772, 2074, 2013, 2108, 2105, 2619, 2040, 14977, 1998]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [34]:
# Add tokenised data in a PyTorch Dataset
# Trainer API requires objects with __getitem__ and __len__
# Each item returns a dict of tensors: input_ids, attention_mask, labels

import torch

class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Build datasets — shared across both versions
train_dataset = EmotionDataset(train_enc, train_labels)
val_dataset   = EmotionDataset(val_enc,   val_labels)
test_dataset  = EmotionDataset(test_enc,  test_labels)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 16000 | Val: 2000 | Test: 2000


In [35]:
# Evaluation metrics (shared across both versions)
# accuracy_score: fraction of correct predictions
# f1 weighted: handles class imbalance by weighting per-class F1 by support
from sklearn.metrics import accuracy_score, f1_score, classification_report

def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
    }

In [36]:
# Method to build a model + trainer for a given config
# Calling separately for v1 and v2 so each version for evaluation which version is good
# Keeping V1 and V2 separate to make it clean and avoid overlapping of weights
from transformers import (
    DistilBertForSequenceClassification,
    TrainingArguments, Trainer
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

def build_trainer(cfg, run_name):
    """Load a fresh DistilBERT and configure Trainer for the given config."""

    # New model from pretrained weights every time
    model = DistilBertForSequenceClassification.from_pretrained(
        model_name,
        num_labels = NUM_LABELS,
        id2label   = id2label,
        label2id   = label2id,
    ).to(DEVICE)

    args = TrainingArguments(
        output_dir                  = f"./results-{cfg['version']}",
        num_train_epochs            = cfg["epochs"],
        per_device_train_batch_size = cfg["batch_size"],
        per_device_eval_batch_size  = 64,
        learning_rate               = cfg["learning_rate"],
        warmup_steps                = cfg["warmup_steps"],
        weight_decay                = cfg["weight_decay"],
        logging_steps               = 50,        # log to W&B every 50 steps
        eval_strategy               = "epoch",   # evaluate after each epoch
        save_strategy               = "epoch",   # checkpoint after each epoch
        load_best_model_at_end      = True,      # restore best checkpoint
        metric_for_best_model       = "f1",      # use F1 to pick best checkpoint
        report_to                   = "wandb",   # stream all metrics to W&B
        run_name                    = run_name,
        fp16                        = torch.cuda.is_available(),  # mixed precision on GPU
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,   # val split used during training
        compute_metrics = compute_metrics,
    )

    return model, trainer

print("build_trainer() is ready")

Device: cuda
build_trainer() is ready


---
## Version 1 — Baseline
`learning_rate=3e-5` &nbsp;|&nbsp; `batch_size=16` &nbsp;|&nbsp; `epochs=3` &nbsp;|&nbsp; `warmup_steps=100`

Conservative learning rate and small batch — standard DistilBERT fine-tuning baseline.

In [42]:
# Version 1: Baseline training
# lr=3e-5 (standard DistilBERT fine-tuning rate)
# batch=16 (conservative — fits all GPU tiers)
# epochs=3 (enough for convergence on this dataset)
# warmup_steps=100 (gradual lr ramp to avoid early instability)

cfg_v1 = {
    "version":       "v1",
    "learning_rate": 3e-5,
    "batch_size":    16,
    "epochs":        3,
    "weight_decay":  0.01,
    "warmup_steps":  100,
    "description":   "Baseline — conservative lr, small batch, 3 epochs",
}

# Initialise W&B run for v1
wandb.init(
    project = "mlops-group-project",
    name    = "distilbert-run-v1",
    config  = {**cfg_v1, "model": model_name, "dataset": "dair-ai/emotion",
               "max_length": max_length, "platform": "Kaggle"},
    reinit  = True,   # allows multiple wandb.init() calls in one notebook session
)

# Build a new model and trainer for v1
model_v1, trainer_v1 = build_trainer(cfg_v1, "distilbert-run-v1")

print(f"   Starting Version 1 training")
print(f"   lr={cfg_v1['learning_rate']}  batch={cfg_v1['batch_size']}  epochs={cfg_v1['epochs']}")

# Train v1
result_v1 = trainer_v1.train()

# Evaluate v1 on test set and log to W&B
eval_v1 = trainer_v1.evaluate(eval_dataset=test_dataset)
wandb.log({
    "final/test_loss":     eval_v1["eval_loss"],
    "final/test_accuracy": eval_v1["eval_accuracy"],
    "final/test_f1":       eval_v1["eval_f1"],
})


print(f"\nVersion 1 complete")
print(f"   Test Accuracy : {eval_v1['eval_accuracy']:.4f}")
print(f"   Test F1       : {eval_v1['eval_f1']:.4f}")
print(f"   Test Loss     : {eval_v1['eval_loss']:.4f}")

eval/accuracy,▁▅█▇▂
eval/f1,▁▅█▇▂
eval/loss,█▄▁▁▂
eval/runtime,█▆▆▆▁
eval/samples_per_second,▁▂▃▃█
eval/steps_per_second,▁▂▃▃█
final/test_accuracy,▁
final/test_f1,▁
final/test_loss,▁
train/epoch,▁▁▂▂▂▂▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇█████
+4,...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   Starting Version 1 training
   lr=3e-05  batch=16  epochs=3


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.199573,0.316610,3,0.926500,0.925977



Version 1 complete
   Test Accuracy : 0.9265
   Test F1       : 0.9260
   Test Loss     : 0.3166


---
## Version 2 — Experiment
`learning_rate=5e-5` &nbsp;|&nbsp; `batch_size=32` &nbsp;|&nbsp; `epochs=4` &nbsp;|&nbsp; `warmup_steps=200`

Higher learning rate, larger batch, one extra epoch — tests whether faster/longer training improves F1.

In [41]:
# Version 2: Experimental training
# lr=5e-5 (higher than v1 — faster convergence, may overfit more)
# batch=32 (larger batch — more stable gradient estimate, needs more GPU memory)
# epochs=4 (extra epoch — gives model more time to learn; risk of overfitting)
# warmup_steps=200 (longer warmup to compensate for higher lr)

cfg_v2 = {
    "version":       "v2",
    "learning_rate": 5e-5,
    "batch_size":    32,
    "epochs":        4,
    "weight_decay":  0.01,
    "warmup_steps":  200,
    "description":   "Experiment — higher lr, larger batch, 4 epochs",
}

# Initialise W&B run for v2 (reinit=True allows second run in same session)
wandb.init(
    project = "mlops-group-project",
    name    = "distilbert-run-v2",
    config  = {**cfg_v2, "model": model_name, "dataset": "dair-ai/emotion",
               "max_length": max_length, "platform": "Kaggle"},
    reinit  = True,
)

# Build a new model from pretrained weights — v2 does NOT inherit v1 weights
model_v2, trainer_v2 = build_trainer(cfg_v2, "distilbert-run-v2")

print(f" Starting Version 2 training")
print(f"   lr={cfg_v2['learning_rate']}  batch={cfg_v2['batch_size']}  epochs={cfg_v2['epochs']}")

# Train v2
result_v2 = trainer_v2.train()

# Evaluate v2 on test set and log to W&B
eval_v2 = trainer_v2.evaluate(eval_dataset=test_dataset)
wandb.log({
    "final/test_loss":     eval_v2["eval_loss"],
    "final/test_accuracy": eval_v2["eval_accuracy"],
    "final/test_f1":       eval_v2["eval_f1"],
})


print(f"\n Version 2 complete")
print(f"   Test Accuracy : {eval_v2['eval_accuracy']:.4f}")
print(f"   Test F1       : {eval_v2['eval_f1']:.4f}")
print(f"   Test Loss     : {eval_v2['eval_loss']:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 Starting Version 2 training
   lr=5e-05  batch=32  epochs=4


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.587690,0.462303,0.925000,0.925986
2,0.316934,0.327979,0.938000,0.937664
3,0.215633,0.235850,0.945500,0.945217
4,0.126467,0.245278,0.943500,0.943303


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.126467,0.265730,4,0.929000,0.928439



 Version 2 complete
   Test Accuracy : 0.9290
   Test F1       : 0.9284
   Test Loss     : 0.2657


---
Comparison & Best Model Selection

In [43]:
# Side-by-side comparison of v1 vs v2
# Picks the best version by test F1 score
# Saves per-class classification report for both versions

results_table = {
    "v1": {
        "accuracy":   eval_v1["eval_accuracy"],
        "f1":         eval_v1["eval_f1"],
        "loss":       eval_v1["eval_loss"],
        "lr":         cfg_v1["learning_rate"],
        "batch":      cfg_v1["batch_size"],
        "epochs":     cfg_v1["epochs"],
        "trainer":    trainer_v1,
        "model":      model_v1,
    },
    "v2": {
        "accuracy":   eval_v2["eval_accuracy"],
        "f1":         eval_v2["eval_f1"],
        "loss":       eval_v2["eval_loss"],
        "lr":         cfg_v2["learning_rate"],
        "batch":      cfg_v2["batch_size"],
        "epochs":     cfg_v2["epochs"],
        "trainer":    trainer_v2,
        "model":      model_v2,
    },
}

# Print comparison table
print("=" * 60)
print(f"  {"Metric":<20} {"Version 1":>15} {"Version 2":>15}")
print("=" * 60)
for metric in ["lr", "batch", "epochs", "accuracy", "f1", "loss"]:
    v1_val = results_table["v1"][metric]
    v2_val = results_table["v2"][metric]
    fmt = ".4f" if isinstance(v1_val, float) and metric not in ["lr"] else ""
    print(f"  {metric:<20} {str(v1_val) if not fmt else format(v1_val, fmt):>15} "
          f"{str(v2_val) if not fmt else format(v2_val, fmt):>15}")
print("=" * 60)

# Pick the best version by F1
best_version = "v1" if eval_v1["eval_f1"] >= eval_v2["eval_f1"] else "v2"
best_trainer = results_table[best_version]["trainer"]
best_model   = results_table[best_version]["model"]
print(f"\n Best version: {best_version} (F1={results_table[best_version]['f1']:.4f})")

# Save per-class classification report for both versions
for ver, res in results_table.items():
    preds  = res["trainer"].predict(test_dataset).predictions.argmax(-1)
    labels = [item["labels"].item() for item in test_dataset]
    report = classification_report(
        labels, preds, target_names=list(id2label.values()), output_dict=True
    )
    with open(f"eval_report_{ver}.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n--- {ver} per-class report ---")
    print(classification_report(labels, preds, target_names=list(id2label.values())))

  Metric                     Version 1       Version 2
  lr                             3e-05           5e-05
  batch                             16              32
  epochs                             3               4
  accuracy                      0.9265          0.9290
  f1                            0.9260          0.9284
  loss                          0.3166          0.2657

 Best version: v2 (F1=0.9284)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- v1 per-class report ---
              precision    recall  f1-score   support

     sadness       0.96      0.98      0.97       581
         joy       0.94      0.96      0.95       695
        love       0.88      0.75      0.81       159
       anger       0.95      0.89      0.92       275
        fear       0.88      0.89      0.88       224
    surprise       0.71      0.79      0.75        66

    accuracy                           0.93      2000
   macro avg       0.89      0.88      0.88      2000
weighted avg       0.93      0.93      0.93      2000



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- v2 per-class report ---
              precision    recall  f1-score   support

     sadness       0.97      0.97      0.97       581
         joy       0.94      0.97      0.95       695
        love       0.92      0.75      0.82       159
       anger       0.91      0.93      0.92       275
        fear       0.90      0.88      0.89       224
    surprise       0.71      0.80      0.75        66

    accuracy                           0.93      2000
   macro avg       0.89      0.88      0.88      2000
weighted avg       0.93      0.93      0.93      2000



In [12]:
import torch

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MyDataset(train_encodings, train_labels_encoded)
test_dataset  = MyDataset(test_encodings,  test_labels_encoded)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Test  dataset: {len(test_dataset)}  samples")
print(f"\nSample item keys: {list(train_dataset[0].keys())}")

Train dataset: 16000 samples
Test  dataset: 2000  samples

Sample item keys: ['input_ids', 'attention_mask', 'labels']


In [47]:
# Cell 13 — Push the best model and tokenizer to Hugging Face Hub
# Only the winning model is published to avoid confusion.
# Ensure HF_REPO is set to a Public repository before submission.
# GitHub Actions inference workflow will pull the model from this URL.

import wandb

# Hugging Face repository (replace with your own username/repo)
HF_REPO = "Maxii2tj/emotion-distilbert"

# Construct Hugging Face model URL
hf_url = f"https://huggingface.co/{HF_REPO}"

print(f" Pushing {best_version} model to {hf_url} ...")

# Push model weights and tokenizer to Hugging Face Hub
best_model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f" Model pushed successfully → {hf_url}")

# Create a final Weights & Biases run to record results
wandb.init(
    project="mlops-group-project",
    name="final-summary",
    reinit=True,
)

# Store model information
wandb.run.summary["best_version"] = best_version
wandb.run.summary["huggingface_model"] = hf_url

# Store evaluation metrics
wandb.run.summary["v1_accuracy"] = eval_v1["eval_accuracy"]
wandb.run.summary["v1_f1"] = eval_v1["eval_f1"]

wandb.run.summary["v2_accuracy"] = eval_v2["eval_accuracy"]
wandb.run.summary["v2_f1"] = eval_v2["eval_f1"]

# Finish W&B run
wandb.finish()

# Final output
print("\n━━━ All Done ━━━")
print(f"Best model : {best_version}")
print(f"HF model   : {hf_url}")
print("W&B project: https://wandb.ai/maxi-i2tj-na/mlops-group-project")
print("GitHub     : https://github.com/aniliter-cloud/ml-ops-group-project/tree/develop")

 Pushing v2 model to https://huggingface.co/Maxii2tj/emotion-distilbert ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


 Model pushed successfully → https://huggingface.co/Maxii2tj/emotion-distilbert


best_version,v2
huggingface_model,https://huggingface....
v1_accuracy,0.9265
v1_f1,0.92598
v2_accuracy,0.929
v2_f1,0.92844



━━━ All Done ━━━
Best model : v2
HF model   : https://huggingface.co/Maxii2tj/emotion-distilbert
W&B project: https://wandb.ai/maxi-i2tj-na/mlops-group-project
GitHub     : https://github.com/aniliter-cloud/ml-ops-group-project/tree/develop
